# Transcript Embedding & Vector Storage Pipeline

**Goal:** To ceate searchable vector embeddings stored in Pinecone.

**Voice Agent Use Cases:**
1. **Summarise sessions** → broad retrieval across summary-level chunks
2. **Answer specific questions** → precise retrieval of detail-level chunks  
3. **"Who said what?"** → speaker-filtered retrieval with attribution
4. **"Take me to where they discuss X"** → timestamp-accurate navigation with deep links

**Pipeline:**
```
Diarized segments (JSON)
    ↓
Multi-resolution chunking (summary + detail)
    ↓  
Context enrichment (embed what you search, store what you display)
    ↓
OpenAI embeddings (text-embedding-3-small, 1536d)
    ↓
Pinecone upsert (event_id namespace, rich metadata)
    ↓
Search & retrieval (intent-based: summary / qa / navigate / who_said)
```

**Prerequisites:**
- Completed diarization notebook with exported `segments.json`
- API keys: OpenAI, Pinecone
- Pinecone index created with dimension=1536, metric=cosine


In [ ]:
# Cell 1 — Install dependencies
# Run once, then restart kernel if needed

# or with uv:
# !uv add openai pinecone

## 1. Configuration

Set your API keys and event metadata. The `SPEAKER_NAMES` mapping converts Pyannote's generic labels (SPEAKER_00, SPEAKER_01) to real names — you'll know these from your event metadata.

The `VIDEO_ID` is your api.video ID, used to generate deep links like `https://embed.api.video/vod/{VIDEO_ID}?t=120` that jump directly to a timestamp.

In [15]:
# Cell 2 — Configuration
import os
from dotenv import load_dotenv

# Load evnironment variables from .env file in project root
load_dotenv()  
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# --- Pinecone settings ---
PINECONE_INDEX_NAME = "avengers"  # must exist, dimension=3072

# --- Event metadata ---
EVENT_ID = "evt_spreadsheet_hacks"         # namespace in Pinecone (one per event)
EVENT_TITLE = "Hacks for Working with Spreadsheets"
VIDEO_ID = "vi4k0jvEUuaTdRAEjQ4Prklg"     # api.video video ID

# --- Speaker mapping (from event metadata / manual identification) ---
SPEAKER_NAMES = {
    "SPEAKER_00": "Victoria",   # Host
    "SPEAKER_01": "David",      # Presenter
}

# --- Embedding settings ---
EMBEDDING_MODEL = "text-embedding-3-large"   # 3072 dimensions
MAX_CHUNK_TOKENS = 300   # target max tokens per detail chunk
MIN_CHUNK_TOKENS = 50    # minimum tokens — below this, merge with neighbour

print("✓ Configuration loaded")

✓ Configuration loaded


## 2. Load Diarized Transcript

The diarization notebook produces a `segments` list where each segment is a continuous block of speech from one speaker:

```python
{
    "start": 0.0,        # seconds
    "end": 12.5,         # seconds  
    "speaker": "SPEAKER_00",
    "text": "Welcome everyone...",
    "word_count": 25
}
```

**Two input options:**
- **Option A:** Load from exported JSON file (recommended for reproducibility)
- **Option B:** Pass the `segments` variable directly if running in the same session

In [6]:
# Cell 3 — Load diarized transcript

import json
from pathlib import Path

SEGMENTS_FILE = "output/hacks-for-working-with-spreadsheets_audio.transcript.json"  # adjust path as needed

segments = None

if Path(SEGMENTS_FILE).exists():
    with open(SEGMENTS_FILE, "r", encoding="utf-8") as f:
        payload = json.load(f)

    # Support both schemas:
    # 1) {"segments": [...], ...}
    # 2) [...]
    if isinstance(payload, dict) and "segments" in payload:
        segments = payload["segments"]
    elif isinstance(payload, list):
        segments = payload
    else:
        raise ValueError(f"Unsupported transcript schema in {SEGMENTS_FILE}")

    if not isinstance(segments, list) or not all(isinstance(s, dict) for s in segments):
        raise TypeError("`segments` must be a list of dict objects")

    print(f"✓ Loaded {len(segments)} segments from {SEGMENTS_FILE}")
else:
    # Option B: If `segments` variable already exists from diarization notebook
    # segments = segments  # uncomment if passing directly
    print(f"⚠ {SEGMENTS_FILE} not found.")
    print("  Either export from diarization notebook with:")
    print('  >>> import json')
    print('  >>> with open("segments.json", "w") as f:')
    print('  >>>     json.dump(segments, f)')
    print("  Or set segments variable directly.")

# Check segment data
if segments:
    total_words = sum(s.get("word_count", len(str(s.get("text", "")).split())) for s in segments)
    speakers = {s.get("speaker", "UNKNOWN") for s in segments}
    duration = segments[-1].get("end", 0)

    print(f"  Segments: {len(segments)}")
    print(f"  Speakers: {speakers}")
    print(f"  Total words: {total_words}")
    print(f"  Duration: {duration/60:.1f} minutes")
    first = segments[0]
    print(f"  First: [{first.get('speaker', 'UNKNOWN')}] {str(first.get('text', ''))[:60]}...")


✓ Loaded 17 segments from output/hacks-for-working-with-spreadsheets_audio.transcript.json
  Segments: 17
  Speakers: {'SPEAKER_01', 'SPEAKER_00'}
  Total words: 4737
  Duration: 30.4 minutes
  First: [SPEAKER_00] Hello, hello and welcome. I will give it a couple of seconds...


## 3. Multi-Resolution Chunking

Create **two granularity levels** from the same data:

| Granularity | Purpose | Size | Agent Use Case |
|------------|---------|------|----------------|
| `summary` | One chunk per speaker turn | Variable (full turn) | "Summarise this session" |
| `detail` | Topic-split within long turns | 50-300 tokens | "What was said about X?", "Take me to..." |

**Why two levels?**
- **Summary chunks** give the agent broad coverage for summarisation — it can retrieve 10 summary chunks and get the gist of the whole video
- **Detail chunks** give precision for Q&A and navigation — when a user asks about "conditional formatting", they get exactly the 30-second segment where it was explained

**Splitting rules for detail chunks:**
1. **Never cross speaker boundaries** — SPEAKER_00 and SPEAKER_01 never share a chunk
2. **Split at topic transitions** — detected by phrases like "the next one is", "moving on to", "so we've got"
3. **Respect token limits** — soft cap at 300 tokens, hard minimum at 50
4. **Merge tiny remainders** — fragments under 50 tokens get absorbed by the previous chunk

In [7]:
# Cell 4 — Multi-resolution chunking

import re

def detect_topic_breaks(text):
    """
    Find sentence indices where the speaker likely shifts topic.
    
    These patterns are tuned for presentation/workshop style speech where
    speakers signal transitions explicitly. For conversational content,
    you might use sentence-embedding similarity drops instead.
    """
    transition_patterns = [
        r"(?:so |okay |now |next |moving on|let's |we're going to|another |the next|here's a|all right)",
        r"(?:we have|we've got|I have|I've got)\s+(?:a |another |some )",
    ]
    sentences = re.split(r'(?<=[.!?])\s+', text)
    breaks = []
    for i, sent in enumerate(sentences):
        for pat in transition_patterns:
            if re.search(pat, sent.lower()) and i > 0:
                breaks.append(i)
                break
    return sentences, breaks


def estimate_tokens(text):
    """Rough token count. OpenAI tokenizer averages ~0.75 words per token for English."""
    return int(len(text.split()) / 0.75)


def build_chunks(segments, speaker_names, max_tokens=300, min_tokens=50):
    """
    Two-level chunking: summary (full speaker turns) + detail (topic splits).
    
    Args:
        segments: list of diarized segments [{start, end, speaker, text}, ...]
        speaker_names: dict mapping speaker IDs to real names
        max_tokens: target maximum tokens per detail chunk
        min_tokens: minimum tokens — below this, merge with previous chunk
    
    Returns:
        list of chunk dicts with granularity field ('summary' or 'detail')
    """
    chunks = []
    chunk_idx = 0
    
    # Step 1: Group consecutive same-speaker segments into turns
    groups = []
    current_group = [segments[0]]
    
    for seg in segments[1:]:
        if seg["speaker"] == current_group[-1]["speaker"]:
            current_group.append(seg)
        else:
            groups.append(current_group)
            current_group = [seg]
    groups.append(current_group)
    
    for group in groups:
        speaker_id = group[0]["speaker"]
        speaker_name = speaker_names.get(speaker_id, speaker_id)
        full_text = " ".join(s["text"] for s in group)
        group_start = group[0]["start"]
        group_end = group[-1]["end"]
        token_est = estimate_tokens(full_text)
        
        # --- Always create a summary chunk for the full turn ---
        chunks.append({
            "text": full_text.strip(),
            "start": float(group_start),
            "end": float(group_end),
            "speaker_id": speaker_id,
            "speaker_name": speaker_name,
            "chunk_index": chunk_idx,
            "granularity": "summary",
        })
        chunk_idx += 1
        
        # --- For long turns, also create detail chunks ---
        if token_est > max_tokens:
            sentences, breaks = detect_topic_breaks(full_text)
            
            if not breaks:
                # No topic transitions detected — fall back to sentence-level splitting
                breaks = list(range(1, len(sentences)))
            
            current_sents = []
            current_tokens = 0
            
            for i, sent in enumerate(sentences):
                sent_tokens = estimate_tokens(sent)
                
                should_flush = (
                    (i in breaks and current_tokens >= min_tokens) or
                    (current_tokens + sent_tokens > max_tokens and current_tokens >= min_tokens)
                )
                
                if should_flush and current_sents:
                    detail_text = " ".join(current_sents).strip()
                    chunks.append({
                        "text": detail_text,
                        "start": float(group_start),
                        "end": float(group_end),
                        "speaker_id": speaker_id,
                        "speaker_name": speaker_name,
                        "chunk_index": chunk_idx,
                        "granularity": "detail",
                    })
                    chunk_idx += 1
                    current_sents = [sent]
                    current_tokens = sent_tokens
                else:
                    current_sents.append(sent)
                    current_tokens += sent_tokens
            
            # Flush remaining sentences
            if current_sents:
                detail_text = " ".join(current_sents).strip()
                if current_tokens >= min_tokens:
                    chunks.append({
                        "text": detail_text,
                        "start": float(group_start),
                        "end": float(group_end),
                        "speaker_id": speaker_id,
                        "speaker_name": speaker_name,
                        "chunk_index": chunk_idx,
                        "granularity": "detail",
                    })
                    chunk_idx += 1
                elif chunks and chunks[-1]["speaker_id"] == speaker_id:
                    # Too small — merge with previous chunk from same speaker
                    chunks[-1]["text"] += " " + detail_text
                    chunks[-1]["end"] = float(group_end)
    
    return chunks


# Run chunking
chunks = build_chunks(segments, SPEAKER_NAMES, MAX_CHUNK_TOKENS, MIN_CHUNK_TOKENS)

summary_chunks = [c for c in chunks if c["granularity"] == "summary"]
detail_chunks = [c for c in chunks if c["granularity"] == "detail"]

print(f"✓ {len(chunks)} total chunks")
print(f"  {len(summary_chunks)} summary (one per speaker turn)")
print(f"  {len(detail_chunks)} detail (topic splits within long turns)")
print()

# Show distribution
for c in chunks[:12]:
    tokens = estimate_tokens(c["text"])
    preview = c["text"][:60].replace("\n", " ")
    print(f"  [{c['chunk_index']:3d}] {c['granularity']:7s} | "
          f"{c['speaker_name']:10s} | ~{tokens:3d} tok | {preview}...")

✓ 63 total chunks
  17 summary (one per speaker turn)
  46 detail (topic splits within long turns)

  [  0] summary | Victoria   | ~914 tok | Hello, hello and welcome. I will give it a couple of seconds...
  [  1] detail  | Victoria   | ~ 56 tok | Hello, hello and welcome. I will give it a couple of seconds...
  [  2] detail  | Victoria   | ~288 tok | It would be great to hear from you all so please do post in ...
  [  3] detail  | Victoria   | ~105 tok | Providing governments with the tools, the skills and the net...
  [  4] detail  | Victoria   | ~100 tok | Now we have Switzerland, many other countries, Wales, Ontari...
  [  5] detail  | Victoria   | ~ 80 tok | The link is in the resources tab at the bottom of your scree...
  [  6] detail  | Victoria   | ~ 81 tok | So I'll get a few technical notes out of the way before we c...
  [  7] detail  | Victoria   | ~ 68 tok | Well now introducing our speaker, this is what we are all he...
  [  8] detail  | Victoria   | ~136 tok | He also ru

## 4. Context Enrichment

**Key principle: Embed what you search, store what you display.**

Raw transcript text like *"the next one is the translate function"* embeds poorly in isolation — it could be about language translation, Excel functions, or programming. By prepending context, we help the embedding model place this chunk closer to relevant queries.

**What we embed** (enriched):
> `"In the session 'Hacks for Working with Spreadsheets', David (presenter) explains: the next one is the translate function..."`

**What we store** (raw):
> `"the next one is the translate function..."`

**What we return to the agent** (raw text + metadata):
> Speaker: David, Time: 16:42, Text: "the next one is the translate function..."

This way the embedding captures the semantic context, but the agent gets clean text to speak back to the user.

In [8]:
# Cell 5 — Context enrichment

def enrich_for_embedding(chunk, event_title):
    """
    Create a context-enriched version of the chunk text for embedding.
    
    The enriched text helps the embedding model understand:
    - What event this is from (helps cross-event search)
    - Who is speaking (helps "who said" queries)
    - What role they have (presenter vs host signals content type)
    - Temporal context (helps ordering and navigation)
    
    Returns the enriched text string. The original chunk['text'] 
    stays unchanged for display.
    """
    minutes = int(chunk["start"] // 60)
    
    if chunk["granularity"] == "summary":
        enriched = (
            f"In the session '{event_title}', "
            f"{chunk['speaker_name']} said: {chunk['text']}"
        )
    else:
        enriched = (
            f"In '{event_title}', "
            f"{chunk['speaker_name']} explains at {minutes} minutes: "
            f"{chunk['text']}"
        )
    
    return enriched


def classify_section(chunk):
    """
    Classify chunk into section type for metadata filtering.
    The agent can use this to narrow retrieval scope.
    """
    text_lower = chunk["text"].lower()
    
    # Intro: first 5 minutes, typically by host
    if chunk["start"] < 300:
        return "introduction"
    # Q&A: questions or interactive parts
    elif "?" in chunk["text"] and len(chunk["text"].split("?")) > 2:
        return "qa"
    # Closing: last 5% of video
    # (you can refine this with actual video duration)
    else:
        return "content"


# Enrich all chunks
for chunk in chunks:
    chunk["embedding_text"] = enrich_for_embedding(chunk, EVENT_TITLE)
    chunk["section"] = classify_section(chunk)

# Preview enriched vs raw
print("=== ENRICHED (what gets embedded) ===")
for c in chunks[:3]:
    print(f"  [{c['granularity']}] {c['embedding_text'][:120]}...")
    print()

print("=== RAW (what gets stored & displayed) ===")
for c in chunks[:3]:
    print(f"  [{c['granularity']}] {c['text'][:120]}...")
    print()

# Section distribution
from collections import Counter
section_counts = Counter(c["section"] for c in chunks)
print(f"Section distribution: {dict(section_counts)}")

=== ENRICHED (what gets embedded) ===
  [summary] In the session 'Hacks for Working with Spreadsheets', Victoria said: Hello, hello and welcome. I will give it a couple o...

  [detail] In 'Hacks for Working with Spreadsheets', Victoria explains at 0 minutes: Hello, hello and welcome. I will give it a cou...

  [detail] In 'Hacks for Working with Spreadsheets', Victoria explains at 0 minutes: It would be great to hear from you all so plea...

=== RAW (what gets stored & displayed) ===
  [summary] Hello, hello and welcome. I will give it a couple of seconds for everybody to join. Well, hello and welcome to this quic...

  [detail] Hello, hello and welcome. I will give it a couple of seconds for everybody to join. Well, hello and welcome to this quic...

  [detail] It would be great to hear from you all so please do post in the chat where you are joining us from and what organization...

Section distribution: {'introduction': 29, 'content': 31, 'qa': 3}


## 5. Generate Embeddings

We use OpenAI's `text-embedding-3-medium` model:
- **8191 token context window** — more than enough for our 300-token chunks

The API accepts batches of up to 100 texts per request. We batch automatically and add a small delay between batches to be kind to rate limits.

In [16]:
# Cell 6 — Generate embeddings with OpenAI

from openai import OpenAI
import time

client = OpenAI(api_key=OPENAI_API_KEY)

def embed_texts(texts, model=EMBEDDING_MODEL, batch_size=100):
    """
    Generate embeddings for a list of texts.
    Handles batching and rate limiting automatically.
    
    Args:
        texts: list of strings to embed
        model: OpenAI embedding model name
        batch_size: texts per API call (max 100)
    
    Returns:
        list of embedding vectors (list[list[float]])
    """
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model=model, input=batch)
        all_embeddings.extend([item.embedding for item in response.data])
        
        # Progress
        done = min(i + batch_size, len(texts))
        print(f"  Embedded {done}/{len(texts)} chunks...", end="\r")
        
        # Rate limit courtesy
        if i + batch_size < len(texts):
            time.sleep(0.1)
    
    print()
    return all_embeddings


# Embed the ENRICHED text (not the raw text)
texts_to_embed = [c["embedding_text"] for c in chunks]

print(f"Embedding {len(texts_to_embed)} chunks with {EMBEDDING_MODEL}...")
start_time = time.time()
embeddings = embed_texts(texts_to_embed)
elapsed = time.time() - start_time

print(f"✓ {len(embeddings)} embeddings generated in {elapsed:.1f}s")
print(f"  Dimension: {len(embeddings[0])}")
print(f"  Approx tokens: ~{sum(estimate_tokens(t) for t in texts_to_embed)}")

Embedding 63 chunks with text-embedding-3-large...
  Embedded 63/63 chunks...
✓ 63 embeddings generated in 1.0s
  Dimension: 3072
  Approx tokens: ~12452


## 6. Store in Pinecone

**Index structure:**
- One Pinecone index (`test-ns`) serves all events
- Each event gets its own **namespace** (e.g. `evt_spreadsheet_hacks`)
- Namespaces are free, unlimited, and isolate data cleanly

**Metadata stored per vector:**
| Field | Type | Purpose |
|-------|------|---------|
| `text` | string | Raw transcript text (for display) |
| `speaker_id` | string | Pyannote label (for filtering) |
| `speaker_name` | string | Human name (for display) |
| `granularity` | string | `summary` or `detail` (for intent routing) |
| `section` | string | `introduction` / `content` / `qa` (for filtering) |
| `start_seconds` | float | Start timestamp (for video navigation) |
| `end_seconds` | float | End timestamp |
| `video_id` | string | api.video ID (for deep link generation) |
| `event_id` | string | Event identifier |
| `chunk_index` | int | Order within the video |

**Vector ID:** Deterministic MD5 hash of `{event_id}:{video_id}:{chunk_index}` — re-running the pipeline overwrites rather than duplicates.

In [17]:
# Cell 7 — Upsert to Pinecone

from pinecone import Pinecone
import hashlib

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

def format_time(seconds):
    """Format seconds as H:MM:SS or M:SS for display."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h}:{m:02d}:{s:02d}" if h > 0 else f"{m}:{s:02d}"


def build_vectors(chunks, embeddings, event_id, video_id):
    """
    Build Pinecone vector dicts with metadata.
    
    Uses deterministic IDs so re-running the pipeline
    updates vectors instead of creating duplicates.
    """
    vectors = []
    
    for chunk, embedding in zip(chunks, embeddings):
        # Deterministic ID
        raw_id = f"{event_id}:{video_id}:{chunk['chunk_index']}"
        vector_id = hashlib.md5(raw_id.encode()).hexdigest()
        
        # Metadata — Pinecone stores this alongside the vector
        # and returns it with search results
        metadata = {
            "text": chunk["text"][:3500],       # Pinecone metadata limit ~40KB
            "speaker_id": chunk["speaker_id"],
            "speaker_name": chunk["speaker_name"],
            "granularity": chunk["granularity"],
            "section": chunk["section"],
            "start_seconds": chunk["start"],
            "end_seconds": chunk["end"],
            "start_time": format_time(chunk["start"]),
            "end_time": format_time(chunk["end"]),
            "video_id": video_id,
            "event_id": event_id,
            "chunk_index": chunk["chunk_index"],
        }
        
        vectors.append({
            "id": vector_id,
            "values": embedding,
            "metadata": metadata,
        })
    
    return vectors


# Build vectors
vectors = build_vectors(chunks, embeddings, EVENT_ID, VIDEO_ID)

# Upsert in batches of 100 (Pinecone limit)
batch_size = 100
for i in range(0, len(vectors), batch_size):
    batch = vectors[i:i + batch_size]
    index.upsert(vectors=batch, namespace=EVENT_ID)
    print(f"  Upserted {min(i + batch_size, len(vectors))}/{len(vectors)}...", end="\r")

print()
print(f"✓ {len(vectors)} vectors upserted to namespace '{EVENT_ID}'")

# Verify
stats = index.describe_index_stats()
ns_stats = stats.get("namespaces", {}).get(EVENT_ID, {})
print(f"  Namespace '{EVENT_ID}': {ns_stats.get('vector_count', 'N/A')} vectors")

  Upserted 63/63...
✓ 63 vectors upserted to namespace 'evt_spreadsheet_hacks'
  Namespace 'evt_spreadsheet_hacks': 63 vectors


## 7. Search & Retrieval

The voice agent will classify user intent and use different search strategies:

| User Says | Intent | Strategy |
|-----------|--------|----------|
| "Summarise this session" | `summary` | Retrieve 10 summary chunks, broad coverage |
| "What was said about conditional formatting?" | `qa` | Retrieve 3-5 detail chunks, high precision |
| "Who talked about data validation?" | `who_said` | Detail chunks + speaker attribution |
| "Take me to the translate function demo" | `navigate` | Detail chunks → return timestamp deep link |

The search function returns results with `video_url` — a direct deep link to that moment in the video.

In [19]:
# Cell 8 — Search with intent-based retrieval

def search_transcripts(
    query,
    intent="qa",
    top_k=5,
    speaker_filter=None,
):
    """
    Semantic search across event transcripts with intent-based routing.
    
    Args:
        query: natural language search query
        intent: one of 'summary', 'qa', 'navigate', 'who_said'
        top_k: number of results to return
        speaker_filter: optional speaker_id to filter by (e.g. "SPEAKER_01")
    
    Returns:
        list of match dicts with text, metadata, score, and video_url
    """
    # 1. Embed the query
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=[query])
    query_vector = response.data[0].embedding
    
    # 2. Build filters based on intent
    filters = {}
    
    if intent == "summary":
        filters["granularity"] = {"$eq": "summary"}
        top_k = max(top_k, 10)  # broad retrieval for summaries
    elif intent in ("qa", "navigate", "who_said"):
        filters["granularity"] = {"$eq": "detail"}
    
    if speaker_filter:
        filters["speaker_id"] = {"$eq": speaker_filter}
    
    # 3. Query Pinecone
    results = index.query(
        vector=query_vector,
        namespace=EVENT_ID,
        top_k=top_k,
        include_metadata=True,
        filter=filters if filters else None,
    )
    
    # 4. Format results
    matches = []
    for match in results.get("matches", []):
        meta = match.get("metadata", {})
        start_sec = int(meta.get("start_seconds", 0))
        
        matches.append({
            "score": match["score"],
            "text": meta.get("text", ""),
            "speaker_name": meta.get("speaker_name", "Unknown"),
            "speaker_id": meta.get("speaker_id", ""),
            "start_time": meta.get("start_time", ""),
            "end_time": meta.get("end_time", ""),
            "start_seconds": meta.get("start_seconds", 0),
            "section": meta.get("section", ""),
            "granularity": meta.get("granularity", ""),
            "video_url": f"https://embed.api.video/vod/{VIDEO_ID}?t={start_sec}",
        })
    
    return matches


def print_results(matches, intent="qa"):
    """Pretty-print search results based on intent."""
    if not matches:
        print("  No results found.")
        return
    
    for i, m in enumerate(matches, 1):
        print(f"\n--- Result {i} (score: {m['score']:.4f}) ---")
        print(f"  Speaker: {m['speaker_name']}")
        print(f"  Time: {m['start_time']} → {m['end_time']} ({m['section']})")
        
        if intent == "navigate":
            print(f"  🔗 {m['video_url']}")
        
        preview = m["text"][:200]
        print(f"  Text: {preview}...")

In [20]:
# Cell 9 — Test search with different intents

print("=" * 60)
print("INTENT: summary — 'Summarise this session'")
print("=" * 60)
results = search_transcripts("summarise the key topics covered", intent="summary", top_k=5)
print_results(results, "summary")

print("\n" + "=" * 60)
print("INTENT: qa — 'What was said about conditional formatting?'")
print("=" * 60)
results = search_transcripts("conditional formatting highlighting cells", intent="qa", top_k=3)
print_results(results, "qa")

print("\n" + "=" * 60)
print("INTENT: navigate — 'Take me to the translate function'")
print("=" * 60)
results = search_transcripts("translate function", intent="navigate", top_k=1)
print_results(results, "navigate")

print("\n" + "=" * 60)
print("INTENT: who_said — 'What did the presenter say about data validation?'")
print("=" * 60)
results = search_transcripts(
    "data validation", 
    intent="who_said", 
    speaker_filter="SPEAKER_01",
    top_k=3,
)
print_results(results, "who_said")

INTENT: summary — 'Summarise this session'

--- Result 1 (score: 0.3502) ---
  Speaker: Victoria
  Time: 29:06 → 30:25 (content)
  Text: well. Thank you for sharing. Thank you all of you and attendees for investing your time in learning. The team is going to pull up a quick poll to find out how you felt about today. And this will help ...

--- Result 2 (score: 0.2792) ---
  Speaker: Victoria
  Time: 22:51 → 23:55 (content)
  Text: Thank you, David. I'm just going to post very quickly the questions we can see. We've had some questions about recording and about the resources. So, please continue sending in your questions. But I j...

--- Result 3 (score: 0.2624) ---
  Speaker: Victoria
  Time: 0:11 → 4:43 (introduction)
  Text: Hello, hello and welcome. I will give it a couple of seconds for everybody to join. Well, hello and welcome to this quickfire masterclass on hacks for working with spreadsheets. My name is Victoria an...

--- Result 4 (score: 0.2599) ---
  Speaker: David
  Time: 2

## 8. Export & Utilities

Export chunks for inspection or use in other tools. The `chunks_export.json` file contains all chunks with their metadata — useful for debugging chunking quality before committing to embeddings.

In [21]:
# Cell 10 — Export chunks for inspection

# Export chunks (without embedding_text to keep file clean)
export_data = []
for c in chunks:
    export_data.append({
        "chunk_index": c["chunk_index"],
        "granularity": c["granularity"],
        "speaker_id": c["speaker_id"],
        "speaker_name": c["speaker_name"],
        "section": c["section"],
        "start": c["start"],
        "end": c["end"],
        "start_time": format_time(c["start"]),
        "end_time": format_time(c["end"]),
        "text": c["text"],
        "token_estimate": estimate_tokens(c["text"]),
    })

with open("chunks_export.json", "w") as f:
    json.dump(export_data, f, indent=2)

print(f"✓ Exported {len(export_data)} chunks to chunks_export.json")

# Summary stats
print(f"\n--- Chunk Statistics ---")
summary_tokens = [estimate_tokens(c["text"]) for c in chunks if c["granularity"] == "summary"]
detail_tokens = [estimate_tokens(c["text"]) for c in chunks if c["granularity"] == "detail"]

if summary_tokens:
    print(f"  Summary chunks: {len(summary_tokens)}")
    print(f"    Token range: {min(summary_tokens)} - {max(summary_tokens)}")
    print(f"    Mean: {sum(summary_tokens)/len(summary_tokens):.0f}")

if detail_tokens:
    print(f"  Detail chunks: {len(detail_tokens)}")
    print(f"    Token range: {min(detail_tokens)} - {max(detail_tokens)}")
    print(f"    Mean: {sum(detail_tokens)/len(detail_tokens):.0f}")

✓ Exported 63 chunks to chunks_export.json

--- Chunk Statistics ---
  Summary chunks: 17
    Token range: 1 - 2256
    Mean: 371
  Detail chunks: 46
    Token range: 50 - 304
    Mean: 114


In [ ]:
# Cell 11 — Utility: delete namespace (re-index from scratch)
# Uncomment to run — this deletes ALL vectors for the event

# index.delete(delete_all=True, namespace=EVENT_ID)
# print(f"✓ Deleted namespace '{EVENT_ID}'")

## Appendix: Exporting from Diarization Notebook

Add this cell at the end of `transcribe_diarize_colab.ipynb` to export segments for this pipeline:

```python
# Cell 12 — Export segments for embedding pipeline
import json

with open("segments.json", "w") as f:
    json.dump(segments, f, indent=2)

print(f"✓ Exported {len(segments)} segments to segments.json")
```

The segments JSON is the contract between the two notebooks. Each segment must have at minimum: `start` (float), `end` (float), `speaker` (string), `text` (string).